# QLoRA fine-tune: resume-JD fit scorer

Distill the Claude teacher's fit scores into Llama-3.1-8B with QLoRA. The student
reproduces the teacher cheaply; it does not beat it.

Needs a T4 runtime and train.jsonl + val.jsonl in `MyDrive/rehearse/`.

## GPU

In [ ]:
import os
# set before anything touches CUDA: cuts fragmentation on the T4
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!nvidia-smi

## Install

transformers/trl/peft pinned (TRL's API shifts between releases). bitsandbytes left
unpinned so it matches Colab's current CUDA: an old pin breaks on the CUDA binary or a
`triton.ops` import.

In [ ]:
!pip install -q -U "transformers==4.46.*" "trl==0.12.*" "peft==0.13.*" \
  bitsandbytes "accelerate==1.1.*" "datasets==3.1.*"

## Hugging Face login

Llama-3.1-8B is gated. Paste the read token.

In [ ]:
from huggingface_hub import login
login()

## Load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/rehearse'
OUT_DIR = '/content/drive/MyDrive/rehearse/adapter'
CKPT_DIR = '/content/drive/MyDrive/rehearse/checkpoints'   # on Drive so it survives a disconnect

from datasets import load_dataset
data = load_dataset('json', data_files={
    'train': f'{DATA_DIR}/train.jsonl',
    'val': f'{DATA_DIR}/val.jsonl',
})
print(data)

## Format

Each row becomes a chat (system + user prompt, teacher JSON as the assistant reply)
built with the model's chat template, so training matches what I send at inference.

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

def to_text(row):
    messages = [
        {'role': 'system', 'content': row['system']},
        {'role': 'user', 'content': row['prompt']},
        {'role': 'assistant', 'content': row['completion']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}

data = data.map(to_text, remove_columns=data['train'].column_names)
print(data['train'][0]['text'][:600])

## Load base model (4-bit)

Llama in 4-bit (nf4), LoRA on the attention and MLP projections. The base stays frozen.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,   # fp16: the T4 (Turing) has no fast bf16
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map='auto'
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## Train

batch 1 with grad-accum 16 (effective 16) and a paged 8-bit optimizer to fit the T4's
~15 GB. Sequence length stays at 1536: the data needs it (shorter truncates the label
JSON at the end of the example). 2 epochs over ~1.3k rows, roughly 1-2h. Checkpoints go
to Drive every 40 steps and the run auto-resumes, so a dropped session costs minutes.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir=CKPT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    logging_steps=10,
    save_strategy='steps',
    save_steps=40,
    save_total_limit=2,
    fp16=True,   # T4 has fp16 tensor cores but not bf16
    max_seq_length=1536,
    dataset_text_field='text',
    report_to='none',
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=data['train'],
    eval_dataset=data['val'],
)
resume = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
trainer.train(resume_from_checkpoint=resume)

## Spot check

Generate on a few val rows and read the JSON. Real evaluation is Phase 3.

In [ ]:
val_raw = load_dataset('json', data_files={'val': f'{DATA_DIR}/val.jsonl'})['val']

def score(system, prompt):
    messages = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt'
    ).to(model.device)
    out = model.generate(inputs, max_new_tokens=400, do_sample=False)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

for ex in val_raw.select(range(3)):
    print('expected:', ex['completion'][:120])
    print('got     :', score(ex['system'], ex['prompt'])[:200])
    print('-' * 60)

## Save adapter

LoRA adapter to Drive (~100-200 MB). The base is re-downloaded when needed.

In [ ]:
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print('saved adapter to', OUT_DIR)

## Predict on the test split

Generate a fit score for every held-out test resume and write them to Drive. Download
predictions.jsonl and score it locally with `python -m finetune.eval_screener`.

In [ ]:
import json, torch

model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

test = [json.loads(l) for l in open(f'{DATA_DIR}/test.jsonl')]
preds = []
for i, ex in enumerate(test):
    ids = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': ex['system']},
         {'role': 'user', 'content': ex['prompt']}],
        add_generation_prompt=True, return_tensors='pt',
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids, attention_mask=torch.ones_like(ids),
            max_new_tokens=400, do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    preds.append({'pair_id': ex['pair_id'], 'raw': raw})
    if (i + 1) % 20 == 0:
        print(f'{i + 1}/{len(test)}')

with open(f'{DATA_DIR}/predictions.jsonl', 'w') as f:
    for p in preds:
        f.write(json.dumps(p) + '\n')
print('wrote', len(preds), 'predictions to', f'{DATA_DIR}/predictions.jsonl')